In [1]:
import numpy as np
import znnl as nl

import pandas as pd

from flax import linen as nn
import optax

import matplotlib.pyplot as plt
from neural_tangents import stax

import h5py as hf

import seaborn as sns

Using backend: cpu

Available hardware:

TFRT_CPU_0

## Download the data

In [ ]:
ensembles = 20
optimizers = ["sgd", "traceopt", "adam"]

for i in range(ensembles):
    generator = nl.data.MNISTGenerator(ds_size=5000)
    seed = np.random.randint(low=0, high=564864168)
    
    for opt in optimizers:
        network = stax.serial(
            stax.Flatten(),
            stax.Dense(128, b_std=0.01, parameterization="standard"),
            stax.Relu(),
            stax.Dense(128, b_std=0.01, parameterization="standard"),
            stax.Relu(),
            stax.Dense(10, b_std=0.01, parameterization="standard"),
        )
        
        if opt == "traceopt":
            optimizer = nl.optimizers.TraceOptimizer(
                scale_factor=1.0, subset=0.3, 
            )
        elif opt == "adam":
            optimizer = optax.adam(1e-3)
        elif opt == "sgd":
            optimizer = optax.sgd(1.0)
        else:
            raise NotImplementedError(
                "Optimizer doesn't exist..."
            )

        model = nl.models.NTModel(
                nt_module=network,
                optimizer=optimizer,
                seed=seed,
                input_shape=(1, 28, 28, 1),
            )

        train_recorder = nl.training_recording.JaxRecorder(
            name=f"{opt}_train_recorder_{i}",
            loss=True,
            accuracy=True,
            update_rate=1
        )
        test_recorder = nl.training_recording.JaxRecorder(
            name=f"{opt}_test_recorder_{i}",
            loss=True,
            accuracy=True,
            update_rate=1
        )

        train_recorder.instantiate_recorder(
            data_set=generator.train_ds
        )
        test_recorder.instantiate_recorder(
            data_set=generator.test_ds
        )

        training_strategy = nl.training_strategies.SimpleTraining(
            model=model, 
            loss_fn=nl.loss_functions.CrossEntropyLoss(),
            accuracy_fn=nl.accuracy_functions.LabelAccuracy(),
            recorders=[train_recorder, test_recorder],
        )
        _ = training_strategy.train_model(
                train_ds=generator.train_ds,
                test_ds=generator.test_ds, 
                epochs=50, 
                batch_size=50,
            )

        train_recorder.dump_records()
        test_recorder.dump_records()

Epoch: 43:  86%|████████████████████████████▍    | 43/50 [10:47<01:40, 14.42s/batch, accuracy=0.916]

In [ ]:
ensembles = 20

def load_data(file, data="loss"):
    with hf.File(file, "r") as db:
        data = db[data][:]
        
    return data

adam_loss_data = []
adam_accuracy_data = []


for i in range(ensembles):
    adam_loss_data.append(
        load_data(f"adam_test_recorder_{i}.h5")
    )
    adam_accuracy_data.append(
        load_data(f"adam_test_recorder_{i}.h5", data="accuracy")
    )
    
traceopt_loss_data = []
traceopt_accuracy_data = []

for i in range(ensembles):
    traceopt_loss_data.append(
        load_data(f"traceopt_test_recorder_{i}.h5")
    )
    traceopt_accuracy_data.append(
        load_data(f"traceopt_test_recorder_{i}.h5", data="accuracy")
    )
    
sgd_loss_data = []
sgd_accuracy_data = []

for i in range(ensembles):
    sgd_loss_data.append(
        load_data(f"sgd_test_recorder_{i}.h5")
    )
    sgd_accuracy_data.append(
        load_data(f"sgd_test_recorder_{i}.h5", data="accuracy")
    )


fig, ax1 = plt.subplots()
ax2 = ax1.twinx()

x = np.linspace(1, 51, 50)

ax1.errorbar(
    x,
    np.mean(adam_loss_data, axis=0), 
    yerr=np.std(adam_loss_data, axis=0),
    marker='o',
    c = "#4EA699",
    mfc="none",
    linestyle="--",
    label="ADAM"
)
ax1.errorbar(
    x,
    np.mean(traceopt_loss_data, axis=0), 
    yerr=np.std(traceopt_loss_data, axis=0),
    marker='x',
    c = "#140D4F",
    mfc="none",
    linestyle="--",
    label="TraceOpt"
)
ax1.errorbar(
    x,
    np.mean(sgd_loss_data, axis=0), 
    yerr=np.std(sgd_loss_data, axis=0),
    marker='*',
    c = "blue",
    mfc="none",
    linestyle="--",
    label="SGD"
)

ax2.errorbar(
    x,
    np.mean(adam_accuracy_data, axis=0), 
    yerr=np.std(adam_accuracy_data, axis=0),
    marker='o',
    c="#4EA699",
    mfc="none",
    linestyle="none",
)
ax2.errorbar(
    x,
    np.mean(traceopt_accuracy_data, axis=0), 
    yerr=np.std(traceopt_accuracy_data, axis=0),
    marker='x',
    c = "#140D4F",
    mfc="none",
    linestyle="none",
)
ax2.errorbar(
    x,
    np.mean(sgd_accuracy_data, axis=0), 
    yerr=np.std(sgd_accuracy_data, axis=0),
    marker='*',
    c = "blue",
    mfc="none",
    linestyle="none",
)

ax1.set_xlabel("Epoch")
ax1.set_ylabel(r"$\mathcal{L}_{test}$")
ax2.set_ylabel(r"Test Accuracy")

# plt.yscale("log")
ax1.set_xscale("log")
fig.legend()
plt.savefig("mnist-dense-vs-adam.pdf")
plt.show()

In [ ]:
with hf.File("adam_test_recorder_0.h5", 'r') as db:
    adata = db["accuracy"][:]
    
with hf.File("adam_test_recorder_1.h5", 'r') as db:
    tdata = db["accuracy"][:]

In [ ]:
plt.plot(adata)
plt.plot(tdata)